# Agentic + RAG metrics: model-family × size matrix (Colab + vLLM)

Runs `tests/agentic_suite/test_live.py` (agentic) and `test_live_rag.py` (RAG judges) against many
open models, each served by vLLM **with its own family's tool-call parser**, then merges the reports
into one comparison per family and size.

What it measures per model:
- native tool calling (vLLM `--tool-call-parser`) and prompt mode (Hermes `<tool_call>` text),
- 16 single-turn cases (single / parallel ×2, ×3, ×4 / mixed tools / dependent / irrelevance /
  enum / numeric / boolean / nested args / distractors / non-English / noisy prompt),
- 4 multi-turn cases through a real agent loop with stub tools,
- `task_completion` judged by the same model (self-judging; see the note at the end),
- the RAG judge metrics (`faithfulness`, `context_precision`, `context_recall`) with the model as the judge,
  on cases with a known right ordering, plus the judge's parse-failure rate.

**GPU tiers** — pick the rows that fit (fp16 weights ≈ 2 GB per billion params, plus KV cache):
T4 16 GB → ≤ 3–4B · L4 24 GB → ≤ 8B · A100 40 GB → ≤ 14B.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

The suite is on branch `test/agentic-rag-suite` (based on `feat/rag-agent-evals`). Clone it with a GitHub
token (the repo is private), or upload a zip of your checkout.

In [ ]:
import os, pathlib
USE_ZIP = False  # True -> upload a zip of your checkout instead of cloning

if USE_ZIP:
    from google.colab import files
    up = files.upload()                       # choose auditkit_agentic_suite.zip
    !rm -rf /content/auditkit && mkdir -p /content/auditkit
    !unzip -q -o {next(iter(up))} -d /content/auditkit
else:
    from getpass import getpass
    tok = getpass("GitHub token: ")
    !git clone -q -b test/agentic-rag-suite https://{tok}@github.com/aryapratinavseth/AuditKIT-Internal.git /content/auditkit
%cd /content/auditkit
!ls

## 2. Install

vLLM brings its own torch. AuditKit's base package has no heavy deps. Llama models are gated:
set `HF_TOKEN` (Colab secrets) or drop those rows.

In [ ]:
!pip -q install vllm
!pip -q install -e . pytest requests
import vllm; print("vLLM", vllm.__version__)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: gated models (Llama) will be skipped")

In [ ]:
# Which tool-call parsers does THIS vLLM build support? (names change between releases)
import subprocess, re
help_txt = subprocess.run(["vllm", "serve", "--help=all"], capture_output=True, text=True).stdout
m = re.search(r"--tool-call-parser[^{]*\{([^}]*)\}", help_txt)
PARSERS = set(p.strip() for p in m.group(1).split(",")) if m else set()
print(sorted(PARSERS) or "could not read parser list; rows will be tried as-is")

## 3. The matrix

Families × sizes. `parser=None` → prompt mode only. `tier` is the smallest GPU that fits.
Edit `RUN_TIERS` to match your runtime.

In [ ]:
MATRIX = [
  # family        hf id                                         size  parser            tier    gated
  ("Qwen2.5",   "Qwen/Qwen2.5-0.5B-Instruct",                   0.5, "hermes",         "t4",   False),
  ("Qwen2.5",   "Qwen/Qwen2.5-1.5B-Instruct",                   1.5, "hermes",         "t4",   False),
  ("Qwen2.5",   "Qwen/Qwen2.5-3B-Instruct",                     3,   "hermes",         "t4",   False),
  ("Qwen2.5",   "Qwen/Qwen2.5-7B-Instruct",                     7,   "hermes",         "l4",   False),
  ("Qwen3",     "Qwen/Qwen3-0.6B",                              0.6, "hermes",         "t4",   False),
  ("Qwen3",     "Qwen/Qwen3-1.7B",                              1.7, "hermes",         "t4",   False),
  ("Qwen3",     "Qwen/Qwen3-4B",                                4,   "hermes",         "l4",   False),
  ("Qwen3",     "Qwen/Qwen3-8B",                                8,   "hermes",         "l4",   False),
  ("Llama-3.2", "meta-llama/Llama-3.2-1B-Instruct",             1,   "llama3_json",    "t4",   True),
  ("Llama-3.2", "meta-llama/Llama-3.2-3B-Instruct",             3,   "llama3_json",    "t4",   True),
  ("Llama-3.1", "meta-llama/Llama-3.1-8B-Instruct",             8,   "llama3_json",    "l4",   True),
  ("Hermes-3",  "NousResearch/Hermes-3-Llama-3.2-3B",           3,   "hermes",         "t4",   False),
  ("Mistral",   "mistralai/Mistral-7B-Instruct-v0.3",           7,   "mistral",        "l4",   False),
  ("Granite",   "ibm-granite/granite-3.3-2b-instruct",          2,   "granite",        "t4",   False),
  ("Granite",   "ibm-granite/granite-3.3-8b-instruct",          8,   "granite",        "l4",   False),
  ("Phi-4",     "microsoft/Phi-4-mini-instruct",                3.8, "phi4_mini_json", "t4",   False),
  ("xLAM",      "Salesforce/Llama-xLAM-2-8b-fc-r",              8,   "xlam",           "l4",   False),
  ("Gemma-3",   "google/gemma-3-4b-it",                         4,   None,             "t4",   True),
  ("SmolLM2",   "HuggingFaceTB/SmolLM2-1.7B-Instruct",          1.7, None,             "t4",   False),
]
RUN_TIERS = {"t4"}          # {"t4"} on a free T4; {"t4","l4"} on L4; {"t4","l4","a100"} on A100
TIER_ORDER = ["t4", "l4", "a100"]
rows = [r for r in MATRIX if r[4] in RUN_TIERS and (not r[5] or os.environ.get("HF_TOKEN"))]
for r in rows: print(r)

## 4. Serve each model, run the suite, stop the server

In [ ]:
import subprocess, time, urllib.request, json, signal, shutil
PORT = 8000
LOG_DIR = pathlib.Path("/content/vllm_logs"); LOG_DIR.mkdir(exist_ok=True)
REPORTS = pathlib.Path("/content/auditkit/tests/agentic_suite/live_reports")
status = {}

def wait_ready(proc, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=2); return True
        except Exception:
            time.sleep(5)
    return False

for family, hf_id, size, parser, tier, gated in rows:
    tag = f"{family}-{size}B".replace(".", "_")
    if parser and PARSERS and parser not in PARSERS:
        print(f"[{tag}] parser {parser!r} not in this vLLM build -> prompt mode only"); parser = None
    cmd = ["vllm", "serve", hf_id, "--port", str(PORT), "--max-model-len", "8192",
           "--gpu-memory-utilization", "0.88", "--dtype", "half"]
    if parser:
        cmd += ["--enable-auto-tool-choice", "--tool-call-parser", parser]
    if family == "Mistral":
        cmd += ["--tokenizer-mode", "mistral", "--config-format", "mistral", "--load-format", "mistral"]
    log = open(LOG_DIR / f"{tag}.log", "w")
    print(f"\n=== {tag}: {' '.join(cmd)}")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    try:
        if not wait_ready(proc):
            status[tag] = "server failed to start (see log)"; print(status[tag]); continue
        env = dict(os.environ, AK_LIVE="1", AK_LIVE_BASE=f"http://localhost:{PORT}",
                   AK_LIVE_MODELS=hf_id, AK_LIVE_NATIVE=hf_id if parser else "",
                   AK_LIVE_JUDGE=hf_id, AK_LIVE_TAG=tag, AK_LIVE_MIN_F1="0.0",
                   AK_LIVE_QUALITY="0", XDG_CACHE_HOME=f"/content/cache/{tag}")
        r = subprocess.run(["python", "-m", "pytest", "-o", "addopts=", "tests/agentic_suite/test_live.py",
                            "tests/agentic_suite/test_live_rag.py", "-q", "-rA", "-p", "no:cacheprovider"],
                           env=env, capture_output=True, text=True)
        print(r.stdout[-2500:])
        status[tag] = "ok" if r.returncode == 0 else f"pytest exit {r.returncode}"
    finally:
        proc.send_signal(signal.SIGINT)
        try: proc.wait(60)
        except Exception: proc.kill()
        log.close(); time.sleep(10)          # let the GPU memory free up
print(status)

## 5. Compare across families and sizes
`AK_LIVE_MIN_F1=0.0` above so weak small models don't abort the report: pipeline assertions still apply.

In [ ]:
import pandas as pd
KEYS = ["tool_call_f1", "tool_call_f1_subset", "tool_call_exact", "tool_call_f1_name", "trajectory_strict",
        "trajectory_strict_subset", "parallel_recall", "parallel_recall_subset", "parallel_precision",
        "parallel_detection", "tool_call_validity", "redundant_tool_calls"]
recs, per_case = [], []
for f in sorted(REPORTS.glob("live_report_*.json")):
    rep = json.loads(f.read_text()); tag = f.stem.replace("live_report_", "")
    for cond, d in rep["conditions"].items():
        mode = cond.split(" / ")[-1]
        recs.append({"model": tag, "mode": mode, **{k: d["headline"].get(k) for k in KEYS}})
        for row in d["rows"]:
            per_case.append({"model": tag, "mode": mode, "case": row["id"], "probe": row["probe"],
                             "f1": row["scores"].get("tool_call_f1"), "par_recall": row["scores"].get("parallel_recall"),
                             "predicted": row["predicted"]})
    for m, d in rep.get("agent", {}).items():
        recs.append({"model": tag, "mode": "agent-loop", **{k: d["headline"].get(k) for k in KEYS + ["task_completion"]}})
summary = pd.DataFrame(recs).set_index(["model", "mode"]).round(3)
display(summary)
cases = pd.DataFrame(per_case)
display(cases.pivot_table(index="case", columns=["model", "mode"], values="f1").round(2))

# RAG judges: parse-failure rate and how many known orderings each judge got right
rag = []
for f in sorted(REPORTS.glob("live_rag_report_*.json")):
    tag = f.stem.replace("live_rag_report_", "")
    for judge, d in json.loads(f.read_text()).items():
        o = d["ordering"]
        rag.append({"model": tag, "parse_error_rate": d["n_errors"] / d["n_calls"],
                    "orderings_right": sum(v is True for v in o.values()), "orderings_total": len(o),
                    **{f"faith:{k}": v for k, v in d["scores"]["faithfulness"].items()},
                    **{f"cp:{k}": v for k, v in d["scores"]["context_precision"].items()},
                    **{f"cr:{k}": v for k, v in d["scores"]["context_recall"].items()}})
display(pd.DataFrame(rag).set_index("model").round(2))

In [ ]:
!cd /content && zip -qr agentic_reports.zip auditkit/tests/agentic_suite/live_reports vllm_logs
from google.colab import files; files.download("/content/agentic_reports.zip")

### Notes
- `AK_LIVE_QUALITY=0` here: a small model getting a RAG ordering wrong is recorded in the table, not a
  failed run. Pipeline checks (no crash, errors recorded instead of zeros) still fail the run.
- **Self-judging**: `task_completion` uses the model under test as its own judge (one GPU, one server).
  Treat it as a pipeline check; for real judge numbers point `AK_LIVE_JUDGE` at a separate, stronger model.
- A model scoring 0 on native mode but fine on prompt mode usually means the **parser** doesn't match the
  model's template, not that the model can't call tools; check `vllm_logs/<tag>.log`.
- Parser names are for current vLLM; the cell in §2 prints what your build supports and unsupported rows
  fall back to prompt mode automatically.